In [1]:
import edsnlp

# Création du pipeline
nlp = edsnlp.blank("fr")
nlp.add_pipe("eds.sentences")
nlp.add_pipe("eds.normalizer")
nlp.add_pipe("eds.dates")
nlp.add_pipe("eds.matcher", config={
    "terms": {
        "cardiologie": ["douleurs thoraciques", "insuffisance cardiaque", "ECG"],
        "neurologie": ["migraine", "AVC", "épilepsie", "céphalées"],
        "pneumologie": ["toux", "dyspnée", "asthme", "bronchite"],
        "symptomes": ["douleur", "fièvre", "fatigue", "nausée"]
    }
})

print("✅ Pipeline EDS-NLP chargé avec succès !")


/Users/nourhtekam/Documents/M2-E3IN BIG Data & IA/Projet LAB/Ctd-IA/ctd-ia/venv/lib/python3.14/site-packages/confit/errors.py:18: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.error_wrappers import (


✅ Pipeline EDS-NLP chargé avec succès !


In [2]:
# Texte médical de test
texte = """
Le patient Jean Dupont, né le 12/03/1965, consulte le 20/01/2024 
pour des douleurs thoraciques depuis 3 jours. 
Il présente également de la fièvre à 38.5°C.
Antécédents : insuffisance cardiaque diagnostiquée en 2020.
"""

# On analyse le texte avec notre pipeline
doc = nlp(texte)

# On affiche les entités détectées
print("📋 Entités détectées :")
for ent in doc.ents:
    print(f"  → '{ent.text}' | Type : {ent.label_}")


📋 Entités détectées :
  → 'douleurs thoraciques' | Type : cardiologie
  → 'fièvre' | Type : symptomes
  → 'insuffisance cardiaque' | Type : cardiologie


In [6]:
import spacy
import re

# Chargement du modèle français
nlp_spacy = spacy.load("fr_core_news_sm")

def anonymiser(texte: str) -> str:
    """Anonymise un texte médical avec spaCy + regex"""
    
    doc = nlp_spacy(texte)
    texte_anonyme = texte
    
    # Remplace les personnes et lieux détectés par spaCy
    for ent in reversed(doc.ents):
        if ent.label_ == "PER":
            texte_anonyme = texte_anonyme.replace(ent.text, "[PATIENT]")
        elif ent.label_ == "LOC":
            texte_anonyme = texte_anonyme.replace(ent.text, "[LIEU]")
        elif ent.label_ == "ORG":
            texte_anonyme = texte_anonyme.replace(ent.text, "[ORGANISATION]")
    
    # Remplace les dates avec regex
    texte_anonyme = re.sub(r'\d{2}/\d{2}/\d{4}', '[DATE]', texte_anonyme)
    texte_anonyme = re.sub(r'\d{2}/\d{4}', '[DATE]', texte_anonyme)
    
    # Anonymiser les années seules (ex: 2020, 1998...)
    texte_anonyme = re.sub(r'\b(19|20)\d{2}\b', '[DATE]', texte_anonyme)

    return texte_anonyme

# Test
print("🔒 Texte anonymisé :")
print(anonymiser(texte))


🔒 Texte anonymisé :

Le patient [PATIENT], né le [DATE], consulte le [DATE] 
pour des douleurs thoraciques depuis 3 jours. 
Il présente également de la fièvre à 38.5°C.
[PATIENT] : insuffisance cardiaque diagnostiquée en [DATE].

